# Batch Machine Learning — Demo Notebook
Run the cells to see batch (offline) training, periodic retraining, and batch scoring.

In [1]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from joblib import dump, load
np.random.seed(42)
OUTDIR = 'outputs'; os.makedirs(OUTDIR, exist_ok=True)

## Example 1 — Batch Regression

In [ ]:
# Number of samples to generate for regression
n = 2000
 # Generate random features for n samples (3 columns: sqft, bedrooms, location)
X = np.random.rand(n, 3)
 # Feature 1: Square footage, scaled between 400 and 3000
sqft = 400 + X[:,0]*2600
 # Feature 2: Number of bedrooms, scaled between 1 and 5 and rounded
bedrooms = (X[:,1]*4 + 1).round()
 # Feature 3: Location score, scaled between 0 and 10
location = X[:,2]*10
 # Target variable: price, calculated using a linear formula plus random noise
price = 30000 + sqft*150 + bedrooms*50000 + location*20000 + np.random.randn(n)*20000
 # Create a DataFrame with all features and target
import pandas as pds
df_reg = pd.DataFrame({'sqft':sqft,'bedrooms':bedrooms,'location_score':location,'price':price})
 # Extract feature matrix and target vector
Xr = df_reg[['sqft','bedrooms','location_score']].values
yr = df_reg['price'].values
 # Split data into training and test sets (80% train, 20% test)
Xr_train,Xr_test,yr_train,yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
 # Train a linear regression model on training data
reg = LinearRegression().fit(Xr_train, yr_train)
 # Predict prices for test set
yr_pred = reg.predict(Xr_test)
 # Print mean squared error and R2 score for model performance
print('MSE:', round(mean_squared_error(yr_test, yr_pred),2), ' R2:', round(r2_score(yr_test, yr_pred),4))

## Example 2 — Batch Classification

In [ ]:
m = 3000
Xp = np.random.randn(m, 4)
tenure = np.abs(Xp[:,0])*36
spend = np.abs(Xp[:,1])*50 + 20
calls = np.abs(Xp[:,2])*3
engage = np.clip(np.abs(Xp[:,3])*5, 0, 10)
logit = -0.03*tenure + 0.02*spend + 0.15*calls - 0.25*engage
prob = 1/(1+np.exp(-logit))
y = (prob>0.5).astype(int)
df_cls = pd.DataFrame({'tenure':tenure,'spend':spend,'calls':calls,'engagement':engage,'churn':y})
Xc = df_cls[['tenure','spend','calls','engagement']].values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xc)
Xc_scaled = scaler.transform(Xc)
Xc_train,Xc_test,yc_train,yc_test = train_test_split(Xc_scaled, y, test_size=0.25, random_state=42)
clf = LogisticRegression(max_iter=200).fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)
print('Accuracy:', round(accuracy_score(yc_test, yc_pred),4))
dump(clf, os.path.join(OUTDIR,'churn_model.joblib')); dump(scaler, os.path.join(OUTDIR,'churn_scaler.joblib'))

## Example 3 — Retrain from Scratch with New Data (Batch)

In [ ]:
m2 = 1500
Xp2 = np.random.randn(m2, 4)
tenure2 = np.abs(Xp2[:,0])*36; spend2 = np.abs(Xp2[:,1])*50 + 20
calls2 = np.abs(Xp2[:,2])*3; engage2 = np.clip(np.abs(Xp2[:,3])*5, 0, 10)
logit2 = -0.03*tenure2 + 0.02*spend2 + 0.15*calls2 - 0.25*engage2 + 0.05
prob2 = 1/(1+np.exp(-logit2)); y2 = (prob2>0.5).astype(int)
df_cls2 = pd.DataFrame({'tenure':tenure2,'spend':spend2,'calls':calls2,'engagement':engage2,'churn':y2})
df_all = pd.concat([df_cls, df_cls2], ignore_index=True)
X_all = df_all[['tenure','spend','calls','engagement']].values
scaler2 = StandardScaler().fit(X_all)
X_all_scaled = scaler2.transform(X_all)
from sklearn.linear_model import LogisticRegression
X_train2,X_test2,y_train2,y_test2 = train_test_split(X_all_scaled, df_all['churn'].values, test_size=0.25, random_state=1)
clf2 = LogisticRegression(max_iter=200).fit(X_train2, y_train2)
print('Retrained accuracy:', round(accuracy_score(y_test2, clf2.predict(X_test2)),4))
dump(clf2, os.path.join(OUTDIR,'churn_model_retrained.joblib')); dump(scaler2, os.path.join(OUTDIR,'churn_scaler_retrained.joblib'))

## Example 4 — Batch Scoring (Bulk Predictions)

In [ ]:
to_score = pd.DataFrame({'tenure':[3,12,24,6,48],'spend':[25,60,40,120,45],'calls':[0,1,2,5,1],'engagement':[9.2,3.1,6.0,1.5,8.0]})
csv_path = os.path.join(OUTDIR,'new_applicants.csv'); to_score.to_csv(csv_path, index=False)
scaler = load(os.path.join(OUTDIR,'churn_scaler_retrained.joblib'))
model = load(os.path.join(OUTDIR,'churn_model_retrained.joblib'))
import numpy as np
preds = model.predict(scaler.transform(to_score.values))
to_score.assign(predicted_churn=preds).to_csv(os.path.join(OUTDIR,'new_applicants_scored.csv'), index=False)
print('Wrote predictions to outputs/new_applicants_scored.csv')